# Resources Generation — Norte Amazónica Bolivia 2025

This notebook generates one `Resources.csv` per cluster (C1–C5) for EnergyScope.

For each cluster, it computes the available local biomass (wood + residues) and exterior resources (diesel, electricity from SIN) by scaling regional totals using each cluster's area or population share.

In [1]:
import unicodedata
import pandas as pd
from pathlib import Path

OUTPUT_DIR = "output_energyscope"

## 1. Cluster definitions

In [2]:
CLUSTERS = {
    1: ["Exaltación", "Reyes", "Santa_Rosa_Beni", "Ixiamas"],
    2: ["Bolpebra"],
    3: ["Guayaramerín", "Riberalta", "Puerto_Gonzalo_Moreno"],
    4: [
        "Bella_Flor", "Filadelfia", "Ingavi", "Nueva_Esperanza", "Porvenir",
        "Puerto_Rico", "San_Lorenzo", "San_Pedro", "Santa_Rosa_Pando",
        "Santos_Mercado", "Sena", "Villa_Nueva",
    ],
    5: ["Cobija"],
}

## 2. Reference values for the whole Norte Amazónica

These regional totals will be split across clusters proportionally:
- **Biomass** → proportional to cluster area (uniform forest density in lowlands)
- **Diesel** → proportional to cluster population (consumption follows households)
- **Electricity from SIN** → fixed per cluster (only C1 has a grid connection)

Sources:
- Biomass: Magne et al. 2024 via EnergyScope Multicell Pathway Bolivia
- Diesel: MHE 2022 departmental energy balance, disaggregated by population
- Electricity SIN: AETN Bolivia Anuario Estadístico 2024 (Reyes 4.49 GWh + Santa Rosa Beni 4.14 GWh + Exaltación ~3.5 GWh; Ixiamas: 0 conservative)

In [3]:
# Regional totals (GWh/year for the whole Norte Amazónica)
WOOD_TOTAL             = 58.388469
BIOMASS_RESIDUES_TOTAL = 226.761768
DIESEL_TOTAL           = 116.4096

# Electricity from SIN — C1 only (SIN line via Trinidad-Caranavi)
# All other clusters run on isolated diesel mini-grids, no SIN connection
ELEC_EXT = {1: 12.13, 2: 0.0, 3: 0.0, 4: 0.0, 5: 0.0}

# Fixed techno-economic parameters per resource (gwp_op_local, c_op_local)
RESOURCE_PARAMS = {
    "WOOD":             (0.011800, 0.016981),
    "BIOMASS_RESIDUES": (0.011800, 0.010318),
    "WASTE":            (0.150100, 0.021188),
    "ELECTRICITY":      (0.412971, 0.063732),
    "GAS":              (0.266600, 0.003808),
    "GASOLINE":         (0.344800, 0.053480),
    "DIESEL":           (0.314800, 0.046630),
    "LFO":              (0.311500, 0.035454),
    "LPG":              (0.283609, 0.021579),
    "LNG":              (0.266600, 0.010663),
    "JET_FUEL":         (0.311500, 0.035454),
}

## 3. Load census data

We read three columns from `CSV_final_in_excel.xlsx` (INE Bolivia Census 2024):
- **Column 3** — municipality name (`MUNICIPIO/TIOC`)
- **Column 4** — area in km² (`SUPERFICIE`)
- **Column 14** — total households 2024 (used as population proxy)

The first 4 rows are multi-level headers, so we skip them.

In [4]:
xlsx_path = Path("../exctraction of data/output/CSV_final_in_excel.xlsx")

df_raw = pd.read_excel(xlsx_path, header=None, skiprows=4)
census = df_raw[[1, 2, 3, 4, 14]].copy()
census.columns = ["DEPTO", "PROVINCIA", "MUNICIPIO", "AREA_km2", "HOUSEHOLDS"]
census = census[census["MUNICIPIO"].notna()].copy()
census["AREA_km2"]   = pd.to_numeric(census["AREA_km2"],   errors="coerce")
census["HOUSEHOLDS"] = pd.to_numeric(census["HOUSEHOLDS"], errors="coerce")

# Normalized name for fuzzy matching (no accents, spaces→underscore, lowercase)
def normalize(name):
    nfkd = unicodedata.normalize("NFKD", name)
    return nfkd.encode("ascii", "ignore").decode("ascii").lower().replace(" ", "_")

census["norm"]      = census["MUNICIPIO"].apply(normalize)
census["depto_norm"] = census["DEPTO"].apply(normalize)

census[["DEPTO", "MUNICIPIO", "AREA_km2", "HOUSEHOLDS"]]

,DEPTO,MUNICIPIO,AREA_km2,HOUSEHOLDS
1,La Paz,Ixiamas,36844,3306
3,Beni,Riberalta,12276,27442
4,Beni,Guayaramerín,5306,10891
5,Beni,Reyes,12377,3417
6,Beni,Santa Rosa,2964,2755
7,Beni,Exaltación,13131,1455
10,Pando,Cobija,740,15564
11,Pando,Porvenir,3180,2230
12,Pando,Bolpebra,7000,802
13,Pando,Bella Flor,1900,1235


## 4. Match municipalities to clusters

Two municipalities share the name **Santa Rosa** — one in Beni (→ C1) and one in Pando (→ C4). We disambiguate by department.

The verification table below shows the area and household count summed per cluster.

In [5]:
def lookup(muni_key):
    """Return (area_km2, households) for a municipality key."""
    norm_key = normalize(muni_key)
    if norm_key == "santa_rosa_beni":
        row = census[(census["norm"] == "santa_rosa") & (census["depto_norm"] == "beni")]
    elif norm_key == "santa_rosa_pando":
        row = census[(census["norm"] == "santa_rosa") & (census["depto_norm"] == "pando")]
    else:
        row = census[census["norm"] == norm_key]
    if row.empty:
        raise KeyError(f"Municipality not found: {muni_key!r}")
    return float(row["AREA_km2"].iloc[0]), float(row["HOUSEHOLDS"].iloc[0])


# Compute per-cluster totals
cluster_area = {}
cluster_pop  = {}

for k, munis in CLUSTERS.items():
    areas, pops = zip(*[lookup(m) for m in munis])
    cluster_area[k] = sum(areas)
    cluster_pop[k]  = sum(pops)

total_area = sum(cluster_area.values())
total_pop  = sum(cluster_pop.values())

# Verification table
rows = [
    {"Cluster": f"C{k}", "Municipalities": len(CLUSTERS[k]),
     "Area (km2)": cluster_area[k], "Households": cluster_pop[k]}
    for k in sorted(CLUSTERS)
]
rows.append({"Cluster": "TOTAL", "Municipalities": sum(len(v) for v in CLUSTERS.values()),
             "Area (km2)": total_area, "Households": total_pop})
pd.DataFrame(rows).set_index("Cluster")

,Municipalities,Area (km2),Households
Cluster,,,
C1,4,65316.0,10933.0
C2,1,7000.0,802.0
C3,3,21482.0,40298.0
C4,12,51772.0,16612.0
C5,1,740.0,15564.0
TOTAL,21,146310.0,84209.0


## 5. Compute per-cluster resource values

| Resource | Scaling rule |
|---|---|
| Wood & biomass residues | proportional to cluster area |
| Diesel (exterior) | proportional to cluster households |
| Electricity (exterior) | fixed values from AETN 2024 |

In [6]:
wood       = {k: WOOD_TOTAL             * (cluster_area[k] / total_area) for k in CLUSTERS}
biomass    = {k: BIOMASS_RESIDUES_TOTAL * (cluster_area[k] / total_area) for k in CLUSTERS}
diesel_ext = {k: DIESEL_TOTAL           * (cluster_pop[k]  / total_pop)  for k in CLUSTERS}

## 6. Generate Resources.csv per cluster

Each file has 11 resource rows and 4 value columns, separated by semicolons.

> **Note on GAS/LNG:** `avail_exterior = 1` is a symbolic Big-M for all clusters.  
> AETN 2024 confirms Norte Amazónica is 100% diesel — no gas or LNG infrastructure exists  
> (Riberalta, Guayaramerín, Cobija all run isolated diesel mini-grids).

In [7]:
BIG_M = 1e15

for k in sorted(CLUSTERS):
    out_path = Path(OUTPUT_DIR) / f"C{k}" / "Resources.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)

    rows = [
        ("WOOD",             wood[k],      BIG_M,         *RESOURCE_PARAMS["WOOD"]),
        ("BIOMASS_RESIDUES", biomass[k],   0.0,           *RESOURCE_PARAMS["BIOMASS_RESIDUES"]),
        ("WASTE",            0.0,          0.0,           *RESOURCE_PARAMS["WASTE"]),
        ("ELECTRICITY",      0.0,          ELEC_EXT[k],   *RESOURCE_PARAMS["ELECTRICITY"]),
        ("GAS",              0.0,          1.0,           *RESOURCE_PARAMS["GAS"]),
        ("GASOLINE",         0.0,          1.0,           *RESOURCE_PARAMS["GASOLINE"]),
        ("DIESEL",           0.0,          diesel_ext[k], *RESOURCE_PARAMS["DIESEL"]),
        ("LFO",              0.0,          1.0,            *RESOURCE_PARAMS["LFO"]),
        ("LPG",              0.0,          1.0,            *RESOURCE_PARAMS["LPG"]),
        ("LNG",              0.0,          1.0,            *RESOURCE_PARAMS["LNG"]),
        ("JET_FUEL",         0.0,          1.0,            *RESOURCE_PARAMS["JET_FUEL"]),
    ]

    lines = [";avail_local;avail_exterior;gwp_op_local;c_op_local"]
    for res, avail_l, avail_e, gwp, c_op in rows:
        lines.append(f"{res};{avail_l:.6f};{avail_e:.6e};{gwp:.6f};{c_op:.6f}")

    out_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

print(f"Saved Resources.csv")

Saved Resources.csv


## 7. Summary

In [8]:
summary = pd.DataFrame([
    {
        "Cluster":           f"C{k}",
        "WOOD (GWh)":        round(wood[k],       4),
        "BIOMASS_RES (GWh)": round(biomass[k],    4),
        "DIESEL_EXT (GWh)": round(diesel_ext[k], 4),
        "ELEC_EXT (GWh)":   ELEC_EXT[k],
    }
    for k in sorted(CLUSTERS)
]).set_index("Cluster")

summary

,WOOD (GWh),BIOMASS_RES (GWh),DIESEL_EXT (GWh),ELEC_EXT (GWh)
Cluster,,,,
C1,26.0659,101.2314,15.1137,12.13
C2,2.7935,10.8491,1.1087,0.00
C3,8.5729,33.2943,55.7075,0.00
C4,20.6608,80.2400,22.9642,0.00
C5,0.2953,1.1469,21.5155,0.00
